# **Generating infant facial images with expressions(angry, cry, happy) using StyleGAN2 ADA**

## **1. Colab Environment Fix**

In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install ninja
!pip install torch torchvision torchaudio

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

## **2. Imports and Setup**

In [2]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch

# Force upfirdn2d to bypass custom CUDA compilation and use PyTorch fallback
!sed -i 's/def _should_use_custom_op():/def _should_use_custom_op():\n    return False/g' /content/stylegan2-ada-pytorch/torch_utils/ops/upfirdn2d.py

# Force bias_act to bypass custom CUDA compilation and use PyTorch fallback
!sed -i 's/def _should_use_custom_op():/def _should_use_custom_op():\n    return False/g' /content/stylegan2-ada-pytorch/torch_utils/ops/bias_act.py

# Force the grid_sample_gradfix script to bypass broken legacy version checks
!sed -i 's/if parse_version(torch.__version__) < parse_version("1.7.0") or LooseVersion(torch.__version__) >= LooseVersion("1.9.0"):/if False:/g' /content/stylegan2-ada-pytorch/torch_utils/ops/grid_sample_gradfix.py

# Force the conv2d_gradfix script to bypass broken legacy version checks
!sed -i 's/if parse_version(torch.__version__) < parse_version("1.7.0") or LooseVersion(torch.__version__) >= LooseVersion("1.9.0"):/if False:/g' /content/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py


import zipfile
import json
import sys
import torch
import torchvision
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("TorchVision version:", torchvision.__version__)

import wandb
wandb.login()

Cloning into 'stylegan2-ada-pytorch'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 131 (delta 0), reused 0 (delta 0), pack-reused 129 (from 2)
Receiving objects: 100% (131/131), 1.13 MiB | 60.93 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/stylegan2-ada-pytorch
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch version: 2.13.0+cu130
TorchVision version: 0.28.0+cu130


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rahuldey-mle (rahuldey-mle-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## **3. Copy Baby Dataset to Workspace**

In [3]:
!cp -r /content/drive/MyDrive/data/baby_samples_gan.zip /content

## **4. Verify Dataset**

In [4]:
zip_path = "/content/baby_samples_gan.zip"

with zipfile.ZipFile(zip_path) as z:
    dataset = json.loads(z.read("dataset.json"))

print("Sample Dataset Labels: \n", dataset["labels"][:3])
print("Dataset length: ", len(dataset["labels"]))

Sample Dataset Labels: 
 [['images (57)_jpg.rf.1BkZDw5o6WCsdd4NmRSY.jpg', 0], ['istockphoto-2070165049-612x612_jpg.rf.MYFhHXTSe5fh1w7SUneI.jpg', 0], ['portrait-serious-stern-baby-600nw-286988390_jpg.rf.LGTpmjbsDCgnnPbMTxZQ.jpg', 0]]
Dataset length:  1200


## **5. Download StyleGAN2 ADA Pretrained Weights for FFHQ**

In [5]:
!wget --content-disposition https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan2/versions/1/files/stylegan2-ffhq-512x512.pkl --output-document stylegan2-ffhq-512x512.pkl

--2026-07-22 23:25:29--  https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan2/versions/1/files/stylegan2-ffhq-512x512.pkl
Resolving api.ngc.nvidia.com (api.ngc.nvidia.com)... 184.33.221.164, 32.185.112.70
Connecting to api.ngc.nvidia.com (api.ngc.nvidia.com)|184.33.221.164|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://xfiles.ngc.nvidia.com/org/nvidia/team/research/models/stylegan2/versions/1/files/stylegan2-ffhq-512x512.pkl?ssec-algo=AES256&versionId=Mb1ieg7MZ5XEpgNpL1K6PP2JxL0Y2hW2&ssec-key=MyPm4i42fsqJoSZdn%2BHofqq5bwxWRlQOptS5i1Z%2B8T72hNFIgfjXmyGBLOX68ZcbwIflgocWP11zDdnPPtoYkx4sdqX3T85UWJODKD%2Bu9Oq9hqahRdKbJuB8%2BeR6aFa1IfD%2Bw60JsxpkwR%2BAnQXOVlh%2FFnQcL8WJLeITgwn3cPW%2FFi7MhtS0rGvysm%2F%2B6XrjPoiVosEmt%2BGM1q9L8DhCURNU50nZTwu%2Fd%2B1AjieQ2Q%2BX1VSDwaAiOOqAVlcxV1Gy2mkatAxrKLn3rrg%2F5PZ6p0q5fRmk%2BrHnofO5Nr1GpLb7t75BN40FcwKcMu4kJC%2F%2FFohFENlinY%2B7umQ7tYdHSnoyDsdAM0pfTb55IRb6Jp%2BWqE1TSJcBTNKd4dCYxiTPncooFj3Xj39qWKFvp0pFX4qTdHFN

## **6. Update Training Loop with W&B Logging**

In [6]:
!cp -r /content/drive/MyDrive/stylegan2ada/training_loop.py /content/stylegan2-ada-pytorch/training/
!cp -r /content/drive/MyDrive/stylegan2ada/misc.py /content/stylegan2-ada-pytorch/torch_utils/   ##this is also a env fix

## **7. Training**

In [ ]:
!python train.py \
    --outdir=/content/drive/MyDrive/stylegan2_baby_runs \
    --data=/content/baby_samples_gan.zip \
    --cond=1 \
    --gpus=1 \
    --cfg=auto \
    --batch=16 \
    --aug=ada \
    --mirror=1 \
    --resume=stylegan2-ffhq-512x512.pkl \
    --snap=25 \
    --kimg=500

Streaming output truncated to the last 5000 lines.
  File "<frozen importlib._bootstrap>", line 1324, in _find_and_load_unlocked
ModuleNotFoundError: No module named 'upfirdn2d_plugin'

  warnings.warn('Failed to build CUDA kernels for upfirdn2d. Falling back to slow reference implementation. Details:\n\n' + traceback.format_exc())
/content/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py:55: UserWarning: conv2d_gradfix not supported on PyTorch 2.13.0+cu130. Falling back to torch.nn.functional.conv2d().
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d().')
/content/stylegan2-ada-pytorch/torch_utils/ops/conv2d_gradfix.py:55: UserWarning: conv2d_gradfix not supported on PyTorch 2.13.0+cu130. Falling back to torch.nn.functional.conv2d().
  warnings.warn(f'conv2d_gradfix not supported on PyTorch {torch.__version__}. Falling back to torch.nn.functional.conv2d().')
Setting up PyTorch plugin "upfirdn2d_plugin"... 